# Fase 3 — FAISS nos recortes antigos

O menu 9 consulta embeddings OpenAI para recortes antigos da mesma conversa e monta um índice FAISS temporário após filtrar empresa, usuário e sessão no SQLite. Este notebook substitui a API por vetores de **fixture manual** para testar o fluxo sem custo; ele não mede a qualidade semântica do modelo real.


In [ ]:
from pathlib import Path
import sys, tempfile
raiz = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(raiz / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.memoria import MemoriaConversas


## Vetores controlados para uma consulta reproduzível
Nesta fixture, `cliente` e `empresa` ocupam a mesma dimensão por definição manual. Em uso real, o modelo de embeddings determina as proximidades.


In [ ]:
class EmbeddingsFixture:
    def vetor(self, texto):
        texto = texto.lower()
        return [1.0 if 'cliente' in texto or 'empresa' in texto else 0.0,
                1.0 if 'orçamento' in texto else 0.0, 0.1]
    def embed_documents(self, textos):
        return [self.vetor(texto) for texto in textos]
    def embed_query(self, texto):
        return self.vetor(texto)


In [ ]:
with tempfile.TemporaryDirectory() as pasta:
    caminho = Path(pasta) / 'memoria.sqlite3'
    ana, bruno = Contexto('demo','ana'), Contexto('demo','bruno')
    with Banco(caminho) as banco:
        banco.preparar_demo()
        memoria = MemoriaConversas(banco)
        sessao = memoria.iniciar(ana)
        memoria.registrar(ana,sessao,'O cliente é Alfa.','[FIXTURE] Entendi.')
        memoria.registrar(ana,sessao,'O orçamento é Beta.','[FIXTURE] Entendi.')
        for numero in range(6):
            memoria.registrar(ana,sessao,f'Assunto {numero}','[FIXTURE] Entendi.')
        achados = memoria.buscar_resumos_vetoriais(ana,sessao,'Qual empresa foi citada?',EmbeddingsFixture(),limite=1)
        assert achados[0]['resumo_extrativo'] == 'O cliente é Alfa.'
        print('FAISS:',achados)
        try:
            memoria.buscar_resumos_vetoriais(bruno,sessao,'Qual empresa?',EmbeddingsFixture())
        except PermissionError:
            print('Outro usuário bloqueado')
        else:
            raise AssertionError('Falha de isolamento')
    with Banco(caminho) as banco:
        memoria = MemoriaConversas(banco)
        assert memoria.buscar_resumos_vetoriais(ana,sessao,'Qual empresa?',EmbeddingsFixture(),limite=1)
        nova = memoria.iniciar(ana)
        assert memoria.buscar_resumos_vetoriais(ana,nova,'Qual empresa?',EmbeddingsFixture()) == []
        print('Reinício preserva recortes; nova conversa não reutiliza recortes antigos.')


## Limites
O índice FAISS é reconstruído em RAM; o SQLite guarda os recortes. O menu 9 usa `text-embedding-3-small` quando a API está disponível e mostra a estratégia usada. Se a API falhar, usa a busca lexical local. Ainda é necessário um teste manual da recuperação com embeddings reais após a janela de seis turnos.
